In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-02-feature-importance.py"

# 同一棵樹，兩種特徵重要性

ESOL 第一次執行會自動下載。需要自行修改或添加分析時，將程式接在寫入分析腳本的儲存格末端，重新執行該格與最末一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ESOL，1,128 筆。水溶解度 log10(mol/L)，DOI 10.1021/ci034243x。
_, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
names = ["MolLogP", "MolWt", "TPSA", "NumRotatableBonds"]
functions = [getattr(Descriptors, name) for name in names]

def describe(dataset):
    mols = [Chem.MolFromSmiles(s) for s in dataset.ids]
    return np.array([[fn(mol) for fn in functions] for mol in mols])

X_train, X_valid = describe(train), describe(valid)
y_train, y_valid = train.y.ravel(), valid.y.ravel()
model = RandomForestRegressor(n_estimators=120, min_samples_leaf=3, random_state=42)
model.fit(X_train, y_train)
prediction = model.predict(X_valid)
print(f"train/valid/test: {len(train)}/{len(valid)}/{len(test)}")
print(f"validation MAE = {mean_absolute_error(y_valid, prediction):.3f}")
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)

from sklearn.inspection import permutation_importance

baseline = mean_absolute_error(y_valid, prediction)
constant = np.full(len(valid), np.median(y_train))
print(f"median baseline MAE = {mean_absolute_error(y_valid, constant):.3f}")
result = permutation_importance(
    model, X_valid, y_valid, scoring="neg_mean_absolute_error",
    n_repeats=30, random_state=42)
for name, mdi, change, sd in zip(names, model.feature_importances_,
                                  result.importances_mean, result.importances_std):
    print(f"{name:18s}  MDI {mdi:.3f}  MAE change {change:+.3f} +/- {sd:.3f}")

tree = model.estimators_[0].tree_
print(f"Tree 1 root: {names[tree.feature[0]]} <= {tree.threshold[0]:.3f}")
for child in [tree.children_left[0], tree.children_right[0]]:
    print(f"weighted count {tree.weighted_n_node_samples[child]:.0f}, "
          f"mean log S {tree.value[child].item():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(8.3, 3.1), sharey=True, layout="constrained")
rows = np.arange(len(names))
axes[0].barh(rows, model.feature_importances_, color="#33768a")
axes[1].barh(rows, result.importances_mean, xerr=result.importances_std,
             capsize=3, color="#bf7245")
axes[0].set_yticks(rows, names)
axes[0].invert_yaxis()
axes[0].set(xlabel="Share of impurity decrease", title="A   Training")
axes[1].set(xlabel="Increase in MAE (log S units)", title="B   Validation")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(out / "11-02-feature-importance.png", dpi=165)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-02-feature-importance.png")))